### 1.AI tool use: 
ChatGPT。<br>
### 2.Task assited: 
提出想做皮卡丘的頭，用圓形做臉、用橢圓形做耳朵，可調部分包括顏色可變、耳朵角度可變，詢問AI還可以做哪些不同的花樣。在AI產生出程式後，詢問特定幾行程式的作用。<br>
### 3.Prompts and interactions: 
某某段程式可以幫我解釋，若解釋的不明白，再問一次AI會用更直白、帶有舉例的說明來回答我的疑問。<br>
### 4.Learning and adaptation: 
原本發想是用兩條拋物線夾出的區域做耳朵，AI建議說若用兩條拋物線會讓圖形變得像杏仁型，因此推薦我使用橢圓形來建立耳朵。在臉頰腮紅顏色可調整部分，原先模型認為我想做的是顏色更改，但我想做的是深紅色到淺紅色的更改，在實際運作發現不對後跟AI回報，詳細說明想要的是紅色色階的改變，AI馬上更改程式。

In [1]:
import numpy as np
from PIL import Image
from IPython.display import display
import ipywidgets as widgets

In [2]:
SIZE = 1024
Y, X = np.ogrid[:SIZE, :SIZE]

BODY_COLORS = {
    0: (255, 218, 42),   # 原版黃
    1: (222, 171, 27),   # 色違深黃
    2: (232, 219, 160),  # 謎擬 Q 風格淺色
}

def circle(cx, cy, radius):
    return (X - cx)**2 + (Y - cy)**2 <= radius**2

def ellipse(cx, cy, rx, ry, angle=0):
    """用座標旋轉，產生傾斜橢圓的 NumPy 遮罩。"""
    theta = np.deg2rad(angle)
    dx, dy = X - cx, Y - cy
    u = dx * np.cos(theta) + dy * np.sin(theta)
    v = -dx * np.sin(theta) + dy * np.cos(theta)
    return (u / rx)**2 + (v / ry)**2 <= 1, v

In [3]:
def draw_pikachu(left_angle=25, right_angle=-25,
                 cheek_depth=50, body_style=0):
    pixels = np.empty((SIZE, SIZE, 3), dtype=np.uint8)
    pixels[:] = (245, 249, 255)

    body_color = BODY_COLORS[body_style]
    dark = (60, 48, 43)

    for cx, angle in [(345, left_angle), (679, right_angle)]:
        outer, _ = ellipse(cx, 263, 78, 240, angle)
        inner, v = ellipse(cx, 263, 71, 233, angle)

        pixels[outer] = dark
        pixels[inner] = body_color
        pixels[inner & (v < -105)] = dark

    pixels[circle(512, 565, 297)] = dark
    pixels[circle(512, 565, 290)] = body_color

    light_red = (255, 180, 185)
    dark_red = (205, 35, 50)
    t = cheek_depth / 100

    cheek_color = tuple(round(light_red[i] * (1 - t) + dark_red[i] * t) for i in range(3))
    pixels[circle(302, 638, 61)] = cheek_color
    pixels[circle(722, 638, 61)] = cheek_color

    for cx in (407, 617):
        eye, _ = ellipse(cx, 514, 26, 35)
        pixels[eye] = dark
        pixels[circle(cx - 8, 501, 8)] = (255, 255, 255)

    nose = (Y >= 608) & (Y <= 627) & (np.abs(X - 512) <= (627 - Y) * 0.85)
    pixels[nose] = dark

    for mouth_cx in (480, 544):
        dx = X - mouth_cx
        curve_y = 660 + 30 * (1 - (dx / 32)**2)
        mouth = (np.abs(dx) <= 32) & (np.abs(Y - curve_y) <= 4)
        pixels[mouth] = dark

    return pixels

In [4]:
def show_pikachu(left_angle, right_angle, cheek_depth, body_style):
    pixels = draw_pikachu(left_angle, right_angle, cheek_depth, body_style)
    display(Image.fromarray(pixels, mode="RGB"))

controls = {
    "left_angle": widgets.IntSlider(
        value=25, min=-45, max=45, description="左耳角度",
        continuous_update=False
    ),
    "right_angle": widgets.IntSlider(
        value=-25, min=-45, max=45, description="右耳角度",
        continuous_update=False
    ),
    "cheek_depth": widgets.IntSlider(
        value=50, min=0, max=100, description="腮紅深淺",
        continuous_update=False
    ),
    "body_style": widgets.IntSlider(
        value=0, min=0, max=2, step=1, description="本體配色",
        continuous_update=False
    ),
}

output = widgets.interactive_output(show_pikachu, controls)
display(widgets.VBox([*controls.values(), output]))